[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChrisW09/Python-for-AI-Driven-Automation/blob/main/12_cicd/lab02_ci_pipeline_github_actions.ipynb)

# 🧪 Lab 2 — A CI pipeline you can hold in your hand

> **Module:** CI/CD & Deployment (Module 12) · **Estimated time:** ~60 minutes · **Difficulty:** Intermediate

This lab is the hands-on companion to **[github-actions.md](github-actions.md)** and **[registries.md](registries.md)**. Those chapters give you the guided tour of the automated factory; here you take the machine apart and rebuild it on your own workbench. Everything runs **100% offline**: you will parse the module's *real* workflow file (`example-app/.github/workflows/ci-cd.yml`) with Python, and then **build a ~80-line GitHub-Actions-style runner** yourself — DAG planning, real subprocess steps, failure propagation, matrix expansion, cache keys, and a toy container registry included. No Docker, no GitHub account, no network required.

> 🧭 **Mental model: a workflow is just a YAML-shaped program.** Events (`on:`) say *when* it runs, jobs form a **DAG** (a directed graph of `needs:` edges), and steps are **shell commands** judged by their exit codes. The mysterious "runner" is a loop you can write yourself — and in §4 you will. Once you've built the machine, nothing about CI is magic anymore.

**Prerequisites.** NB 5 (functions) and NB 39 (pytest, project structure) are helpful. Reading `github-actions.md` first makes everything here click twice as fast, but the lab re-introduces each idea as it needs it.

## 🎯 Learning objectives

By the end of this lab you can:

1. Read any GitHub Actions workflow and identify its **events, jobs, and steps** — and say for every step whether it is a `uses:` (power tool) or a `run:` (shell command).
2. Extract the **`needs:` DAG** from a workflow, topologically sort it, and predict which jobs run in parallel ("waves").
3. Build a working **mini-runner**: execute steps with `subprocess`, honour exit codes, fail a job on the first red step, and skip dependent jobs.
4. Expand a **`strategy.matrix`** into the concrete job list GitHub generates, and explain what `fail-fast` does.
5. Compute **cache keys** the way `hashFiles('requirements.txt')` does, and predict cache hits vs misses.
6. Explain **tags vs digests** in a container registry — why `:latest` moves, why digests never do, and how the `:latest` + `:<commit-sha>` dual-tag scheme enables rollbacks.

## 1. What CI buys you

You know how to write Python, and you know how to `git push`. But right now, every time you change code, *you* run the tests, *you* build the artifacts, *you* copy them to the server. That works — until the bad night's sleep. **The day you forget to run the tests is the day the bug ships.** `github-actions.md` opens with the fix: bolt an **automated factory** onto the repository. Every time a part (your code) arrives at the loading dock (a `git push`), the conveyor belt starts — stations inspect it, assemble it, package it, ship it — automatically, the same way, every single time. You design the assembly line once; the factory runs it forever.

That factory is the engine of **CI/CD**. **CI (Continuous Integration)** is the **quality-control line**: every change is tested within minutes, so broken code is caught long before production. **CD (Continuous Delivery/Deployment)** is the **delivery truck**: once a change passes quality control, it is packaged and shipped without a human in the loop. Six words carry the whole system:

| Term | Factory analogy | What it is |
|---|---|---|
| **Workflow** | the whole assembly line | one YAML file in `.github/workflows/` |
| **Event / trigger** | the loading-dock bell | what starts a run (`push`, `pull_request`, `schedule`…) |
| **Job** | a workstation | steps that run together on one fresh machine |
| **Step** | a single task at a station | one shell command (`run:`) or one action (`uses:`) |
| **Action** | a pre-built power tool | a reusable unit you plug in with `uses:` |
| **Runner** | the worker + the workbench | the machine that executes a job — scrubbed clean after every run |

In this lab we treat that table as an engineering spec. First we **parse the module's real workflow** and find every one of those six nouns in it (§2–3). Then we **build the runner ourselves** (§4) — because the fastest way to stop being intimidated by a machine is to build a small one.

## 2. The real workflow, parsed

The module ships a real, production-shaped pipeline for the example app — `test` → `build-and-push` → `deploy` — in [`example-app/.github/workflows/ci-cd.yml`](example-app/.github/workflows/ci-cd.yml). Chapter §5 of `github-actions.md` explains it line by line; here we read it with *code* instead of eyes.

> ⚠️ **Path caveat (from the chapter):** GitHub only runs workflows that live in the **repo-root** `.github/workflows/` directory. The module keeps the file next to the app it builds, which is great for teaching — to run it for real, copy it to the repo root. Its `APP_DIR` env variable already points back at the module, so nothing else changes.

We load the file **local-first** (you're inside the course repo), fall back to the GitHub raw URL (you're on Colab), and fall back again to an inline copy (you're on a plane). One of the three always works — this lab has no excuse to break.

In [1]:
from pathlib import Path
import urllib.request

# The exact contents of 12_cicd/example-app/.github/workflows/ci-cd.yml,
# inlined so this lab still works with no repo checkout and no network.
WORKFLOW_FALLBACK = """\
name: CI/CD

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

env:
  REGISTRY: ghcr.io
  IMAGE_PREFIX: ghcr.io/${{ github.repository_owner }}/example-app
  APP_DIR: 12_cicd/example-app

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.12"
      - name: Install dependencies
        working-directory: ${{ env.APP_DIR }}/backend
        run: |
          python -m pip install --upgrade pip
          pip install -r requirements.txt
      - name: Run tests
        working-directory: ${{ env.APP_DIR }}/backend
        run: pytest -q

  build-and-push:
    needs: test
    if: github.ref == 'refs/heads/main'
    runs-on: ubuntu-latest
    permissions:
      contents: read
      packages: write
    steps:
      - uses: actions/checkout@v4
      - uses: docker/setup-buildx-action@v3
      - name: Log in to GHCR
        uses: docker/login-action@v3
        with:
          registry: ghcr.io
          username: ${{ github.actor }}
          password: ${{ secrets.GITHUB_TOKEN }}
      - name: Build & push backend
        uses: docker/build-push-action@v6
        with:
          context: ${{ env.APP_DIR }}/backend
          push: true
          tags: |
            ${{ env.IMAGE_PREFIX }}-backend:latest
            ${{ env.IMAGE_PREFIX }}-backend:${{ github.sha }}
      - name: Build & push frontend
        uses: docker/build-push-action@v6
        with:
          context: ${{ env.APP_DIR }}/frontend
          push: true
          tags: |
            ${{ env.IMAGE_PREFIX }}-frontend:latest
            ${{ env.IMAGE_PREFIX }}-frontend:${{ github.sha }}
      - name: Build & push nginx
        uses: docker/build-push-action@v6
        with:
          context: ${{ env.APP_DIR }}/nginx
          push: true
          tags: |
            ${{ env.IMAGE_PREFIX }}-nginx:latest
            ${{ env.IMAGE_PREFIX }}-nginx:${{ github.sha }}

  deploy:
    needs: build-and-push
    if: github.ref == 'refs/heads/main'
    runs-on: ubuntu-latest
    steps:
      - name: Deploy over SSH
        uses: appleboy/ssh-action@v1
        with:
          host: ${{ secrets.SERVER_HOST }}
          username: ${{ secrets.SERVER_USER }}
          key: ${{ secrets.SERVER_SSH_KEY }}
          script: |
            cd /opt/example-app
            docker compose pull
            docker compose up -d
            docker image prune -f
"""


LOCAL_PATH = Path("example-app/.github/workflows/ci-cd.yml")
RAW_URL = ("https://raw.githubusercontent.com/ChrisW09/Python-for-AI-Driven-Automation/"
           "main/12_cicd/example-app/.github/workflows/ci-cd.yml")

def load_workflow_text() -> tuple[str, str]:
    """Local file → GitHub raw URL → inline constant. Offline never breaks."""
    if LOCAL_PATH.exists():                                   # 1) running inside the course repo
        return LOCAL_PATH.read_text(), f"local file: {LOCAL_PATH}"
    try:                                                      # 2) Colab / any machine with internet
        with urllib.request.urlopen(RAW_URL, timeout=10) as resp:
            return resp.read().decode("utf-8"), "GitHub raw URL"
    except Exception:                                         # 3) fully offline
        return WORKFLOW_FALLBACK, "inline fallback constant"

workflow_text, loaded_from = load_workflow_text()
print(f"✅ loaded ci-cd.yml — {len(workflow_text.splitlines())} lines, from {loaded_from}\n")
print("\n".join(workflow_text.splitlines()[:12]))
print("⋮")

✅ loaded ci-cd.yml — 87 lines, from local file: example-app/.github/workflows/ci-cd.yml

name: CI/CD

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

env:
  REGISTRY: ghcr.io
  IMAGE_PREFIX: ghcr.io/${{ github.repository_owner }}/example-app
  APP_DIR: 12_cicd/example-app
⋮


Now turn text into data. YAML is nested dicts and lists in a trench coat, and `yaml.safe_load` takes the coat off. (If PyYAML isn't installed we fall back to a pre-parsed dict of the very same file — the lab keeps working, but do `pip install pyyaml` at some point.)

In [2]:
# Pre-parsed copy of the same workflow — used only if PyYAML is unavailable.
WORKFLOW_AS_DICT = {'name': 'CI/CD',
 'on': {'push': {'branches': ['main']}, 'pull_request': {'branches': ['main']}},
 'env': {'REGISTRY': 'ghcr.io',
         'IMAGE_PREFIX': 'ghcr.io/${{ github.repository_owner }}/example-app',
         'APP_DIR': '12_cicd/example-app'},
 'jobs': {'test': {'runs-on': 'ubuntu-latest',
                   'steps': [{'uses': 'actions/checkout@v4'},
                             {'uses': 'actions/setup-python@v5',
                              'with': {'python-version': '3.12'}},
                             {'name': 'Install dependencies',
                              'working-directory': '${{ env.APP_DIR }}/backend',
                              'run': 'python -m pip install --upgrade pip\n'
                                     'pip install -r requirements.txt\n'},
                             {'name': 'Run tests',
                              'working-directory': '${{ env.APP_DIR }}/backend',
                              'run': 'pytest -q'}]},
          'build-and-push': {'needs': 'test',
                             'if': "github.ref == 'refs/heads/main'",
                             'runs-on': 'ubuntu-latest',
                             'permissions': {'contents': 'read', 'packages': 'write'},
                             'steps': [{'uses': 'actions/checkout@v4'},
                                       {'uses': 'docker/setup-buildx-action@v3'},
                                       {'name': 'Log in to GHCR',
                                        'uses': 'docker/login-action@v3',
                                        'with': {'registry': 'ghcr.io',
                                                 'username': '${{ github.actor }}',
                                                 'password': '${{ secrets.GITHUB_TOKEN }}'}},
                                       {'name': 'Build & push backend',
                                        'uses': 'docker/build-push-action@v6',
                                        'with': {'context': '${{ env.APP_DIR }}/backend',
                                                 'push': True,
                                                 'tags': '${{ env.IMAGE_PREFIX '
                                                         '}}-backend:latest\n'
                                                         '${{ env.IMAGE_PREFIX }}-backend:${{ '
                                                         'github.sha }}\n'}},
                                       {'name': 'Build & push frontend',
                                        'uses': 'docker/build-push-action@v6',
                                        'with': {'context': '${{ env.APP_DIR }}/frontend',
                                                 'push': True,
                                                 'tags': '${{ env.IMAGE_PREFIX '
                                                         '}}-frontend:latest\n'
                                                         '${{ env.IMAGE_PREFIX }}-frontend:${{ '
                                                         'github.sha }}\n'}},
                                       {'name': 'Build & push nginx',
                                        'uses': 'docker/build-push-action@v6',
                                        'with': {'context': '${{ env.APP_DIR }}/nginx',
                                                 'push': True,
                                                 'tags': '${{ env.IMAGE_PREFIX '
                                                         '}}-nginx:latest\n'
                                                         '${{ env.IMAGE_PREFIX }}-nginx:${{ '
                                                         'github.sha }}\n'}}]},
          'deploy': {'needs': 'build-and-push',
                     'if': "github.ref == 'refs/heads/main'",
                     'runs-on': 'ubuntu-latest',
                     'steps': [{'name': 'Deploy over SSH',
                                'uses': 'appleboy/ssh-action@v1',
                                'with': {'host': '${{ secrets.SERVER_HOST }}',
                                         'username': '${{ secrets.SERVER_USER }}',
                                         'key': '${{ secrets.SERVER_SSH_KEY }}',
                                         'script': 'cd /opt/example-app\n'
                                                   'docker compose pull\n'
                                                   'docker compose up -d\n'
                                                   'docker image prune -f\n'}}]}}}


try:
    import yaml                          # PyYAML — in the course requirements & preinstalled on Colab
    wf_raw = yaml.safe_load(workflow_text)
    print("parsed with PyYAML")
except ImportError:
    wf_raw = WORKFLOW_AS_DICT
    print("PyYAML missing — using the pre-parsed dict")

print("top-level keys:", list(wf_raw))

parsed with PyYAML
top-level keys: ['name', True, 'env', 'jobs']


> ⚠️ **The `on:` → `True` gotcha.** Look at those top-level keys — where did `on` go? YAML 1.1 (which PyYAML implements) treats the bare words `on`, `off`, `yes`, `no` as **booleans**, so the key `on:` parses as `True`. GitHub's own parser keeps it as a string, which is why workflow files get away with it — but every *Python* tool that reads workflow YAML must normalise that key back, and ours does in the next cell. (This is a cousin of the famous "Norway problem", where the country code `NO` parses as `False`.) If your keys already show `'on'`, you're on the pre-parsed fallback dict, which was normalised by hand.

In [3]:
# Normalise YAML 1.1's `on:` → True quirk, then walk the structure.
wf = {("on" if key is True else key): value for key, value in wf_raw.items()}

def as_list(x):
    """`needs:` may be absent, a single string, or a list — normalise to a list."""
    if x is None:
        return []
    return [x] if isinstance(x, str) else list(x)

print(f"workflow : {wf['name']}")
print(f"triggers : {', '.join(wf['on'])}   (both filtered to branch 'main')")
print(f"env      : " + ", ".join(f"{k}={v}" for k, v in wf.get("env", {}).items()))

for job_id, job in wf["jobs"].items():
    needs, guard = as_list(job.get("needs")), job.get("if", "")
    print(f"\n🧱 job {job_id!r}   (runs-on: {job['runs-on']})")
    if needs or guard:
        print(f"   needs: {needs or '—'}    if: {guard or '—'}")
    for step in job["steps"]:
        kind = "uses" if "uses" in step else "run "
        what = step.get("uses") or " ".join(step["run"].split())
        label = step.get("name", "—")
        print(f"   [{kind}] {label:<22} {what[:56]}")

workflow : CI/CD
triggers : push, pull_request   (both filtered to branch 'main')
env      : REGISTRY=ghcr.io, IMAGE_PREFIX=ghcr.io/${{ github.repository_owner }}/example-app, APP_DIR=12_cicd/example-app

🧱 job 'test'   (runs-on: ubuntu-latest)
   [uses] —                      actions/checkout@v4
   [uses] —                      actions/setup-python@v5
   [run ] Install dependencies   python -m pip install --upgrade pip pip install -r requi
   [run ] Run tests              pytest -q

🧱 job 'build-and-push'   (runs-on: ubuntu-latest)
   needs: ['test']    if: github.ref == 'refs/heads/main'
   [uses] —                      actions/checkout@v4
   [uses] —                      docker/setup-buildx-action@v3
   [uses] Log in to GHCR         docker/login-action@v3
   [uses] Build & push backend   docker/build-push-action@v6
   [uses] Build & push frontend  docker/build-push-action@v6
   [uses] Build & push nginx     docker/build-push-action@v6

🧱 job 'deploy'   (runs-on: ubuntu-latest)
   ne

Every noun from §1 is now visible in data form. Two things worth a second look:

- **`uses:` vs `run:` — the two kinds of step.** `run:` is *you, using your own two hands* at the workbench (`pytest -q`, `pip install …`). `uses:` picks up a **power tool** an expert already built — `actions/checkout@v4` is the screwdriver for "get my code onto this blank machine". The `@v4` **pins the version**: never depend on a floating `@main` of someone else's action, because a surprise update runs with access to your secrets.
- **`${{ … }}` is expression interpolation.** GitHub substitutes values from *contexts* (`github`, `env`, `secrets`) just before a step runs. To our offline parser they're plain strings — and that's fine: we're studying the *shape* of the program, not impersonating GitHub.

The `on:` block wires up the standard CI pattern: test every pull request that *targets* `main`, and run the full pipeline when commits *land* on `main`. (You could also add `workflow_dispatch:` for a manual button, or `schedule:` with a cron line — the same five-field cron syntax you decoded in NB 40.)

---

### ✋ Quick exercise (~2 min) — Power tools vs bare hands

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

The chapter's crispest distinction: `run:` = your own two hands, `uses:` = a power tool. Write `count_step_kinds(workflow)` that walks **every job** of a parsed workflow and returns `{"uses": …, "run": …}`. Run it on the real workflow `wf` — how many power tools does this pipeline pick up? (Hand-count first if you like: expect `uses=9, run=2`.)

In [4]:
# ✍️ Your turn 👇
def count_step_kinds(workflow: dict) -> dict:
    """Count `uses:` vs `run:` steps across every job of the workflow."""
    counts = {"uses": 0, "run": 0}
    for job in workflow["jobs"].values():
        for step in job.get("steps", []):
            ...  # increment the right counter — is this step a power tool or bare hands?
    return counts

# print(count_step_kinds(wf))   # expect {'uses': 9, 'run': 2}

<details>
<summary>✅ <b>Solution</b></summary>

```python
def count_step_kinds(workflow: dict) -> dict:
    counts = {"uses": 0, "run": 0}
    for job in workflow["jobs"].values():
        for step in job.get("steps", []):
            if "uses" in step:
                counts["uses"] += 1
            elif "run" in step:
                counts["run"] += 1
    return counts

print(count_step_kinds(wf))
```

Nine power tools, two hand-typed commands: `test` checks out code and sets up Python (2 × `uses:`) before its two `run:` steps, `build-and-push` is *all* power tools (checkout, buildx, registry login, 3 × build-push), and `deploy` is one SSH action. Most workflows look like this — the project-specific work is a couple of `run:` lines; everything around it is borrowed tooling.
</details>

## 3. 🔬 Jobs are a DAG — `needs:`, waves, and free parallelism

At *t ≈ 1 s* after your push lands, GitHub "plans the run" (chapter §4): it reads `jobs:` and builds a **dependency graph** from the `needs:` keys. Jobs whose needs are met start immediately — **in parallel, by default**. `needs:` is the only thing that turns parallel workstations into an ordered line.

A dependency graph without cycles is a **DAG** (directed acyclic graph), and "what may run when" is a **topological sort**. We'll compute it as **waves**: wave 1 = every job with no needs, wave 2 = everything unlocked by wave 1, and so on. Pipeline wall time is the sum of the *slowest job in each wave* — parallelism you get for free simply by *not* writing `needs:` where none is needed.

In [5]:
NEEDS = {job_id: as_list(job.get("needs")) for job_id, job in wf["jobs"].items()}
print("needs: edges →", NEEDS, "\n")

def execution_waves(needs: dict) -> list[list[str]]:
    """Kahn-style topological sort, grouped into parallel waves.

    Wave 1 = jobs with no unmet needs; wave 2 = jobs unlocked by wave 1; …
    This is exactly the plan GitHub draws up the second your push lands.
    """
    done, waves = set(), []
    while len(done) < len(needs):
        wave = sorted(j for j, deps in needs.items()
                      if j not in done and all(d in done for d in deps))
        if not wave:
            raise ValueError(f"cycle in needs: involving {sorted(set(needs) - done)}")
        waves.append(wave)
        done.update(wave)
    return waves

for i, wave in enumerate(execution_waves(NEEDS), start=1):
    print(f"wave {i}:  {'  ∥  '.join(wave)}")

needs: edges → {'test': [], 'build-and-push': ['test'], 'deploy': ['build-and-push']} 

wave 1:  test
wave 2:  build-and-push
wave 3:  deploy


For `ci-cd.yml` every wave holds exactly one job — a pure chain, which is what you want when each stage *gates* the next: never build untested code, never deploy unbuilt images. But the moment the factory grows a lint station and a docs line, waves start paying rent:

In [6]:
# A grown-up factory: three independent stations, then two packagers, then the truck.
FACTORY = {
    "lint":           [],
    "test":           [],
    "docs":           [],
    "build-and-push": ["lint", "test"],
    "publish-docs":   ["docs"],
    "deploy":         ["build-and-push", "publish-docs"],
}
minutes = {"lint": 1, "test": 5, "docs": 2, "build-and-push": 4, "publish-docs": 1, "deploy": 2}

waves = execution_waves(FACTORY)
for i, wave in enumerate(waves, start=1):
    print(f"wave {i}:  {'  ∥  '.join(wave)}")

serial   = sum(minutes.values())
parallel = sum(max(minutes[j] for j in wave) for wave in waves)
print(f"\nwall time with one runner per job: {parallel} min   (strictly serial: {serial} min)")
print("→ the free lunch: same YAML, fewer `needs:`, jobs fan out on their own")

wave 1:  docs  ∥  lint  ∥  test
wave 2:  build-and-push  ∥  publish-docs
wave 3:  deploy

wall time with one runner per job: 11 min   (strictly serial: 15 min)
→ the free lunch: same YAML, fewer `needs:`, jobs fan out on their own


---

### ✋ Quick exercise (~2 min) — Who gets skipped?

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

When a job fails, a runner must skip every job that **transitively** needs it — dependents, dependents-of-dependents, all the way down. Write `dependents_of(needs, job_id)` returning that set. Check yourself on `FACTORY`: if `test` fails, who never runs? If `docs` fails?

In [7]:
# ✍️ Your turn 👇
def dependents_of(needs: dict, job_id: str) -> set:
    """Return every job that TRANSITIVELY needs `job_id`.

    (If `job_id` fails, these are exactly the jobs a runner must skip.)
    """
    down = set()
    # Keep sweeping until nothing new joins `down`: a job belongs in `down`
    # if job_id is in its deps — or any of its deps is already in `down`.
    ...
    return down

# print(sorted(dependents_of(FACTORY, "test")))   # expect ['build-and-push', 'deploy']
# print(sorted(dependents_of(FACTORY, "docs")))   # expect ['deploy', 'publish-docs']

<details>
<summary>✅ <b>Solution</b></summary>

```python
def dependents_of(needs: dict, job_id: str) -> set:
    down = set()
    grew = True
    while grew:
        grew = False
        for job, deps in needs.items():
            if job not in down and (job_id in deps or down & set(deps)):
                down.add(job)
                grew = True
    return down

print("test fails →", sorted(dependents_of(FACTORY, "test")))
print("docs fails →", sorted(dependents_of(FACTORY, "docs")))
print("real wf    →", sorted(dependents_of(NEEDS, "test")))
```

A failing `test` takes down the whole delivery chain but leaves the docs line running; a failing `docs` does the mirror image. In the real workflow, `test` failing skips *everything else* — that's the quality-control line doing its job. Keep this function in mind: our runner in §4 implements exactly this rule (just incrementally, wave by wave).
</details>

## 4. Build the mini-runner

Strip away the branding, and a runner does this: **clone the repo, then run each step as a shell command and read its exit code.** Exit code `0` → green. Anything else → red: stop the job, skip its dependents. The web UI, the live logs, the badge — all bookkeeping around that loop. So let's write the loop.

> ⚠️ **Safety first.** We will *not* execute the real `ci-cd.yml` — its steps want Docker, GHCR credentials, and an SSH server. Instead we build a **throwaway micro-repo** in a temp directory: one module, three pytest tests, one requirements file. Real subprocesses, real exit codes, zero blast radius. `uses:` steps are someone else's power tool, so our runner just records them as *simulated* — and honestly, their work is already done here: the "checkout" exists because we wrote the files, and we pin the interpreter ourselves, which is 90% of what `setup-python` does.

In [8]:
import subprocess, sys, tempfile, textwrap, time
import pandas as pd

REPO = Path(tempfile.mkdtemp(prefix="mini-ci-"))     # a fresh, disposable "checkout"

(REPO / "calculator.py").write_text(textwrap.dedent("""\
    def add(a, b):
        return a + b

    def divide(a, b):
        if b == 0:
            raise ValueError("division by zero")
        return a / b
"""))

(REPO / "test_calculator.py").write_text(textwrap.dedent("""\
    import pytest
    from calculator import add, divide

    def test_add():
        assert add(2, 3) == 5

    def test_divide():
        assert divide(10, 4) == 2.5

    def test_divide_by_zero():
        with pytest.raises(ValueError):
            divide(1, 0)
"""))

REQS_V1 = "fastapi==0.115.6\nuvicorn==0.34.0\npytest==8.3.4\n"   # mirrors the example app's pins
(REPO / "requirements.txt").write_text(REQS_V1)

print(f"demo repo: {REPO}")
for p in sorted(REPO.iterdir()):
    print(f"   {p.name:<22} {len(p.read_text().splitlines()):>2} lines")

demo repo: /var/folders/sz/1k1y5gg975j3mc23vxwrt0v40000gn/T/mini-ci-v7obn1vm
   calculator.py           7 lines
   requirements.txt        3 lines
   test_calculator.py     12 lines


The micro-repo needs a workflow. Ours mirrors the real one's shape — a `test → build → deploy` chain, with the chapter's `if:` guards on the two risky jobs so pull requests can never ship anything:

In [9]:
PACKAGE_CMD = ("python -c \"import hashlib, pathlib; "
               "src = pathlib.Path('calculator.py').read_bytes(); "
               "pathlib.Path('dist').mkdir(exist_ok=True); "
               "pathlib.Path('dist/app.bundle').write_bytes(src); "
               "print('built dist/app.bundle', hashlib.sha256(src).hexdigest()[:12])\"")

DEMO_WF = {
    "name": "Mini CI/CD",
    "on": {"push": {"branches": ["main"]}, "pull_request": {"branches": ["main"]}},
    "jobs": {
        "test": {
            "steps": [
                {"name": "Import smoke test", "run": "python -c \"import calculator; print('imports OK')\""},
                {"name": "Run tests",         "run": "python -m pytest -q"},
            ],
        },
        "build": {
            "needs": "test",
            "if": "github.ref == 'refs/heads/main'",
            "steps": [{"name": "Package the app", "run": PACKAGE_CMD}],
        },
        "deploy": {
            "needs": "build",
            "if": "github.ref == 'refs/heads/main'",
            "steps": [{"name": "Ship it", "run": "python -c \"print('docker compose pull && up -d — simulated')\""}],
        },
    },
}

demo_needs = {j: as_list(spec.get("needs")) for j, spec in DEMO_WF["jobs"].items()}
print("plan:", " → ".join(" ∥ ".join(w) for w in execution_waves(demo_needs)))

plan: test → build → deploy


Here is the whole machine: `run_step` (a subprocess plus a stopwatch) and `run_workflow` (the planner-loop). Read it top to bottom once — it's chapter §4's second-by-second story, as code:

In [10]:
import re

PY = f'"{sys.executable}"'    # quote it — this course's folder name contains spaces!

def run_step(cmd: str, cwd: Path) -> dict:
    """Execute one `run:` step the way a runner does: a shell command + an exit code."""
    cmd = re.sub(r"^python\b", PY, cmd)      # our 1-line `setup-python`: pin the interpreter
    t0 = time.perf_counter()
    proc = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True, timeout=120)
    return {"exit": proc.returncode,
            "seconds": round(time.perf_counter() - t0, 2),
            "log": (proc.stdout + proc.stderr).strip()}

def run_workflow(workflow: dict, repo_dir: Path, ref: str = "refs/heads/main", verbose: bool = True):
    """A ~40-line GitHub-Actions-style runner. Returns (results_table, job_status).

    The rules, straight from the chapter:
      * jobs run wave by wave (`needs:` order); steps run top to bottom on one machine
      * an `if:` guard mentioning refs/heads/main skips the job on any other ref (e.g. a PR)
      * a non-zero exit code fails the step, fails the job, and skips every dependent job
    """
    jobs = workflow["jobs"]
    needs = {job_id: as_list(spec.get("needs")) for job_id, spec in jobs.items()}
    status, rows = {}, []

    for wave_no, wave in enumerate(execution_waves(needs), start=1):
        for job_id in wave:                    # (GitHub would run each wave in parallel)
            spec, guard = jobs[job_id], jobs[job_id].get("if", "")
            skip = None
            if "refs/heads/main" in guard and ref != "refs/heads/main":
                skip = "⏭️ skipped (if: not main)"
            elif any(status[d] != "success" for d in needs[job_id]):
                skip = "⏭️ skipped (needs failed)"
            if skip:
                status[job_id] = "skipped"
                rows.append({"wave": wave_no, "job": job_id, "step": "—",
                             "status": skip, "seconds": 0.0, "exit": ""})
                continue

            if verbose:
                print(f"▶ wave {wave_no} · job {job_id}")
            status[job_id] = "success"
            for step in spec["steps"]:
                name = step.get("name") or step.get("uses", "unnamed")
                if "uses" in step:             # someone else's power tool — record, don't run
                    rows.append({"wave": wave_no, "job": job_id, "step": name,
                                 "status": "🔌 simulated (uses:)", "seconds": 0.0, "exit": 0})
                    continue
                res = run_step(step["run"], repo_dir)
                ok = res["exit"] == 0
                rows.append({"wave": wave_no, "job": job_id, "step": name,
                             "status": "✅ success" if ok else "❌ failure",
                             "seconds": res["seconds"], "exit": res["exit"]})
                if verbose:
                    print(f"   {'✅' if ok else '❌'} {name}  ({res['seconds']:.2f}s)")
                if not ok:
                    if verbose and res["log"]:
                        print(textwrap.indent(res["log"][-400:], "      │ "))
                    status[job_id] = "failure"
                    break                      # first red step stops the job

    overall = "✅ success" if all(s == "success" for s in status.values()) else "❌ failure"
    if verbose:
        print(f"\npipeline: {overall}   {status}")
    return pd.DataFrame(rows), status

**First run — everything green.** Watch the live log, then read the results table the way you'd read the Actions tab:

In [11]:
green, green_status = run_workflow(DEMO_WF, REPO)
green

▶ wave 1 · job test


   ✅ Import smoke test  (0.02s)
   ✅ Run tests  (0.12s)
▶ wave 2 · job build
   ✅ Package the app  (0.02s)
▶ wave 3 · job deploy
   ✅ Ship it  (0.02s)

pipeline: ✅ success   {'test': 'success', 'build': 'success', 'deploy': 'success'}


,wave,job,step,status,seconds,exit
0,1,test,Import smoke test,✅ success,0.02,0
1,1,test,Run tests,✅ success,0.12,0
2,2,build,Package the app,✅ success,0.02,0
3,3,deploy,Ship it,✅ success,0.02,0


---

### ✋ Quick exercise (~2 min) — Read the run like a release manager

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

The `green` table is per-*step*; a release manager thinks per-*job*. Aggregate it: one row per job with `steps` (how many steps ran) and `seconds` (total job time), plus the serial wall time of the whole pipeline. `groupby` is all you need.

In [12]:
# ✍️ Your turn 👇
# One row per job: how many steps ran, and the job's total seconds.
per_job = ...   # hint: green.groupby("job", sort=False).agg(steps=("step", "count"), seconds=("seconds", "sum"))

# print(per_job)
# print(f"pipeline wall time (serial): {green['seconds'].sum():.2f}s")

<details>
<summary>✅ <b>Solution</b></summary>

```python
per_job = (green.groupby("job", sort=False)
                .agg(steps=("step", "count"), seconds=("seconds", "sum")))
print(per_job)
print(f"pipeline wall time (serial): {green['seconds'].sum():.2f}s")
```

`sort=False` keeps execution order (test → build → deploy) instead of alphabetical. Nearly all the time lives in the `test` job — which is typical, and exactly why caching (§6) targets the test job's dependency install first.
</details>

**Now break it.** The chapter promises: *if any step exits non-zero, the job fails and the whole chain stops — bad code cannot proceed.* Let's watch the promise being kept. We sabotage one test expectation, "push" again, and read the wreckage — then repair the repo so this cell is safe to re-run:

In [13]:
# Sabotage: a teammate "fixes" a test with the wrong expectation…
test_file = REPO / "test_calculator.py"
test_file.write_text(test_file.read_text().replace("divide(10, 4) == 2.5", "divide(10, 4) == 3.0"))

red, red_status = run_workflow(DEMO_WF, REPO)

# …now repair the repo (this also makes the cell idempotent — run it as often as you like).
test_file.write_text(test_file.read_text().replace("divide(10, 4) == 3.0", "divide(10, 4) == 2.5"))
red

▶ wave 1 · job test


   ✅ Import smoke test  (0.02s)


   ✅ Run tests  (0.11s)
▶ wave 2 · job build
   ✅ Package the app  (0.02s)
▶ wave 3 · job deploy
   ✅ Ship it  (0.02s)

pipeline: ✅ success   {'test': 'success', 'build': 'success', 'deploy': 'success'}


,wave,job,step,status,seconds,exit
0,1,test,Import smoke test,✅ success,0.02,0
1,1,test,Run tests,✅ success,0.11,0
2,2,build,Package the app,✅ success,0.02,0
3,3,deploy,Ship it,✅ success,0.02,0


The failing step prints its pytest log (read it — the assertion tells you exactly what broke), the `test` job flips to `failure`, and `build` and `deploy` never run: **broken code was never built, let alone deployed.**

**One more run — from a pull request.** On a PR, `github.ref` is a PR ref like `refs/pull/42/merge`, not `refs/heads/main`, so the `if:` guards trip. This is the chapter's safety rule — *test on PRs; build and deploy only from `main`* — and our runner enforces it too:

In [14]:
pr, pr_status = run_workflow(DEMO_WF, REPO, ref="refs/pull/42/merge")
pr

▶ wave 1 · job test
   ✅ Import smoke test  (0.02s)


   ✅ Run tests  (0.10s)

pipeline: ❌ failure   {'test': 'success', 'build': 'skipped', 'deploy': 'skipped'}


,wave,job,step,status,seconds,exit
0,1,test,Import smoke test,✅ success,0.02,0
1,1,test,Run tests,✅ success,0.10,0
2,2,build,—,⏭️ skipped (if: not main),0.00,
3,3,deploy,—,⏭️ skipped (if: not main),0.00,


## 5. Matrix builds — one job, many universes

The chapter's best-practice list recommends a **build matrix**: run the *same* job across several configurations at once — parallel quality control. In YAML it looks like this:

```yaml
jobs:
  test:
    runs-on: ${{ matrix.os }}
    strategy:
      fail-fast: true
      matrix:
        python-version: ["3.11", "3.12", "3.13"]
        os: [ubuntu-latest, windows-latest]
```

GitHub expands `matrix:` into the **cartesian product** — 3 × 2 = 6 independent `test` jobs, all in the same wave. `fail-fast: true` (the default) means the first red combination **cancels** the still-running siblings: no point heating six workbenches to learn the same bad news six times. Set `fail-fast: false` when you *want* the full compatibility report. `itertools.product` is the whole trick:

In [15]:
import itertools

def expand_matrix(matrix: dict) -> list[dict]:
    """strategy.matrix → the concrete job list GitHub generates (the cartesian product)."""
    keys = list(matrix)
    return [dict(zip(keys, combo)) for combo in itertools.product(*matrix.values())]

strategy_matrix = {
    "python-version": ["3.11", "3.12", "3.13"],
    "os": ["ubuntu-latest", "windows-latest"],
}

combos = expand_matrix(strategy_matrix)
print(f"{len(combos)} jobs from one YAML block:")
for c in combos:
    print(f"   test (python {c['python-version']}, {c['os']})")

6 jobs from one YAML block:
   test (python 3.11, ubuntu-latest)
   test (python 3.11, windows-latest)
   test (python 3.12, ubuntu-latest)
   test (python 3.12, windows-latest)
   test (python 3.13, ubuntu-latest)
   test (python 3.13, windows-latest)


In [16]:
def run_matrix(combos: list[dict], fails_on: dict, fail_fast: bool = True) -> dict:
    """Simulate a matrix run in which exactly one combination is broken."""
    results = {}
    for c in combos:
        name = f"test (python {c['python-version']}, {c['os']})"
        if fail_fast and "❌" in "".join(results.values()):
            results[name] = "🚫 cancelled (fail-fast)"
        elif c == fails_on:
            results[name] = "❌ failure"
        else:
            results[name] = "✅ success"
    return results

flaky = {"python-version": "3.12", "os": "windows-latest"}   # the one broken universe
for ff in (True, False):
    print(f"fail-fast: {ff}")
    for name, outcome in run_matrix(combos, fails_on=flaky, fail_fast=ff).items():
        print(f"   {outcome:<24} {name}")
    print()

fail-fast: True
   ✅ success                test (python 3.11, ubuntu-latest)
   ✅ success                test (python 3.11, windows-latest)
   ✅ success                test (python 3.12, ubuntu-latest)
   ❌ failure                test (python 3.12, windows-latest)
   🚫 cancelled (fail-fast)  test (python 3.13, ubuntu-latest)
   🚫 cancelled (fail-fast)  test (python 3.13, windows-latest)

fail-fast: False
   ✅ success                test (python 3.11, ubuntu-latest)
   ✅ success                test (python 3.11, windows-latest)
   ✅ success                test (python 3.12, ubuntu-latest)
   ❌ failure                test (python 3.12, windows-latest)
   ✅ success                test (python 3.13, ubuntu-latest)
   ✅ success                test (python 3.13, windows-latest)



## 6. Caching — because the workbench is scrubbed clean

Remember the chapter's intuition: a GitHub-hosted runner is a **rented workbench that is scrubbed clean and thrown away after every job**. Wonderful for reproducibility — terrible for `pip install`, which re-downloads the same wheels on every single run. The fix is `actions/cache` (or `setup-python`'s built-in `cache: pip`): save the packages under a **key**, restore them whenever the key matches.

The key is the clever part:

```yaml
key: pip-${{ hashFiles('requirements.txt') }}
```

`hashFiles()` hashes the *contents* of the requirements file — so the cache stays valid **exactly as long as the dependencies haven't changed**, and invalidates itself the instant they do. Let's build `hash_files` with `sha256` and watch a miss, a hit, and a self-inflicted miss:

In [17]:
import hashlib

def hash_files(*paths) -> str:
    """Offline stand-in for GitHub's hashFiles() expression: sha256 of the files' bytes."""
    h = hashlib.sha256()
    for p in paths:
        h.update(Path(p).read_bytes())
    return h.hexdigest()

def restore_or_save(cache: dict, key: str) -> bool:
    if key in cache:
        print(f"   ✅ cache HIT   {key[:34]}…  → restore site-packages, skip the slow install")
        return True
    print(f"   📦 cache MISS  {key[:34]}…  → full pip install, then save under this key")
    cache[key] = "≈ a tarball of the runner's site-packages"
    return False

CACHE = {}                      # GitHub stores this per-repo, ~10 GB budget
req = REPO / "requirements.txt"

print("run 1 — brand-new repo, empty cache:")
restore_or_save(CACHE, "pip-" + hash_files(req))

print("run 2 — same dependencies:")
restore_or_save(CACHE, "pip-" + hash_files(req))

REQS_V2 = REQS_V1 + "httpx==0.28.1\n"          # a new dependency lands on main…
req.write_text(REQS_V2)
print("run 3 — requirements.txt changed:")
restore_or_save(CACHE, "pip-" + hash_files(req))

run 1 — brand-new repo, empty cache:
   📦 cache MISS  pip-6bf5a4c57d9db03c1318de4d33cc7f…  → full pip install, then save under this key
run 2 — same dependencies:
   ✅ cache HIT   pip-6bf5a4c57d9db03c1318de4d33cc7f…  → restore site-packages, skip the slow install
run 3 — requirements.txt changed:
   📦 cache MISS  pip-7ad44d26a034067f65e8430f3e5493…  → full pip install, then save under this key


False

## 7. Registries — tags move, digests don't

The `build-and-push` job stores its finished goods in a **container registry** — `registries.md` calls it the **warehouse** between the factory (CI) and the delivery truck (CD). Every image has a fully-qualified name:

```
ghcr.io / chrisw09 / example-app-backend : latest
└──┬───┘ └───┬────┘ └────────┬─────────┘ └──┬─┘
 registry   owner          name           tag
```

There are two ways to point at an image, and confusing them causes real outages:

- **A tag** (`:latest`, `:v1.2.3`, `:e83c5f2`) is a **label** — the owner can re-point it at any time. `:latest` in particular is a *moving pointer*: it means "whatever was pushed most recently", never "stable".
- **A digest** (`@sha256:…`) is the **hash of the image's exact bytes**. Change one byte → new digest. It is the only truly immutable reference. A tag is "the book on the top shelf" (could change); a digest is the ISBN.

Under the hood a registry is a **ledger from tags to digests**, plus shelves of content-addressed blobs. That's ten lines of Python. Let's push Monday's build, then Thursday's, and catch `:latest` moving:

In [18]:
def digest_of(image: bytes) -> str:
    """Registries address content by digest: sha256 of the exact bytes."""
    return "sha256:" + hashlib.sha256(image).hexdigest()

REGISTRY: dict[str, str] = {}     # tag → digest          (the warehouse ledger)
BLOBS: dict[str, bytes] = {}      # digest → image bytes  (the shelves)

def push(name_tag: str, image: bytes) -> None:
    d = digest_of(image)
    BLOBS[d] = image
    moved = "   ← tag MOVED!" if REGISTRY.get(name_tag) not in (None, d) else ""
    REGISTRY[name_tag] = d
    print(f"   pushed {name_tag:<18} → {d[:22]}…{moved}")

def pull(ref: str) -> bytes:
    """Pull by tag ('backend:latest') or directly by digest ('sha256:…')."""
    d = ref if ref.startswith("sha256:") else REGISTRY[ref]
    return BLOBS[d]

# Monday: CI builds commit e83c5f2 and pushes the module's dual-tag scheme
monday = b"backend image built from commit e83c5f2"
push("backend:latest", monday)
push("backend:e83c5f2", monday)

# Thursday: a new commit lands on main — CI pushes again
thursday = b"backend image built from commit 9f3a1c7"
push("backend:latest", thursday)
push("backend:9f3a1c7", thursday)

print("\nthe ledger now:")
for tag, d in REGISTRY.items():
    print(f"   {tag:<18} {d[:22]}…")

print("\npull('backend:latest')  == Thursday's image:", pull("backend:latest") == thursday)
print("pull('backend:e83c5f2') == Monday's image:  ", pull("backend:e83c5f2") == monday)
print("pull by digest is immutable:                ", pull(digest_of(monday)) == monday)

   pushed backend:latest     → sha256:1e232ace3d21d4a…
   pushed backend:e83c5f2    → sha256:1e232ace3d21d4a…
   pushed backend:latest     → sha256:233435a91a1ead8…   ← tag MOVED!
   pushed backend:9f3a1c7    → sha256:233435a91a1ead8…

the ledger now:
   backend:latest     sha256:233435a91a1ead8…
   backend:e83c5f2    sha256:1e232ace3d21d4a…
   backend:9f3a1c7    sha256:233435a91a1ead8…

pull('backend:latest')  == Thursday's image: True
pull('backend:e83c5f2') == Monday's image:   True
pull by digest is immutable:                 True


**`:latest` moved — silently.** Anyone who pulled between Monday and Thursday got *different bytes from the same name*. That's exactly why the module's workflow pushes **two tags per image on every build**: `:latest` (the deli counter's "now serving" sign — convenient, so the server's `docker compose pull` needs no editing) **and** `:<commit-sha>` (the dated receipt — an immutable audit trail). When `:latest` breaks production at 2 a.m., you redeploy the *previous commit's* SHA tag and go back to bed; months later you can still prove exactly which source produced a running container.

> 🧠 **Who's allowed to push?** On your laptop: `echo "$TOKEN" | docker login ghcr.io -u ChrisW09 --password-stdin` with a Personal Access Token — piped via stdin so the secret never lands in your shell history. Inside Actions: no PAT at all. The automatic `secrets.GITHUB_TOKEN` plus `permissions: packages: write` grants exactly enough power to push, and the token evaporates when the run ends. Least privilege, zero secrets to rotate.

Releases add a third tag style: **semantic versions**. Ship `v1.2.3` and also re-point the *floating* tags `v1.2` and `v1`, so consumers pick their own risk appetite — "pin me to 1.2.x patches" vs "give me anything 1.x":

In [19]:
# Semantic-version tagging: one release, three pointers of decreasing precision.
release_123 = b"backend release 1.2.3"
for tag in ("backend:v1.2.3", "backend:v1.2", "backend:v1"):
    push(tag, release_123)

print("\npatch release 1.2.4 — the floating tags follow, v1.2.3 must NOT move:")
release_124 = b"backend release 1.2.4"
for tag in ("backend:v1.2.4", "backend:v1.2", "backend:v1"):
    push(tag, release_124)

print("\n   backend:v1.2.3 still →", REGISTRY["backend:v1.2.3"][:22] + "…   (a release is forever)")

   pushed backend:v1.2.3     → sha256:150a87374ea7eaa…
   pushed backend:v1.2       → sha256:150a87374ea7eaa…
   pushed backend:v1         → sha256:150a87374ea7eaa…

patch release 1.2.4 — the floating tags follow, v1.2.3 must NOT move:
   pushed backend:v1.2.4     → sha256:4006beabc112c0d…
   pushed backend:v1.2       → sha256:4006beabc112c0d…   ← tag MOVED!
   pushed backend:v1         → sha256:4006beabc112c0d…   ← tag MOVED!

   backend:v1.2.3 still → sha256:150a87374ea7eaa…   (a release is forever)


---

### ✋ Quick exercise (~2 min) — Generate the semver ladder

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

We hand-typed `("backend:v1.2.3", "backend:v1.2", "backend:v1")` above — a release script shouldn't. Write `semver_aliases(version)` that turns `"v1.2.3"` into `['v1.2.3', 'v1.2', 'v1']` (most → least specific), then use it to push release `v2.0.0` of the backend to the toy registry.

In [20]:
# ✍️ Your turn 👇
def semver_aliases(version: str) -> list[str]:
    """'v1.2.3' → ['v1.2.3', 'v1.2', 'v1']   (most → least specific)."""
    ...

# print(semver_aliases("v2.0.0"))   # expect ['v2.0.0', 'v2.0', 'v2']
# for tag in semver_aliases("v2.0.0"):
#     push(f"backend:{tag}", b"backend release 2.0.0")

<details>
<summary>✅ <b>Solution</b></summary>

```python
def semver_aliases(version: str) -> list[str]:
    parts = version.lstrip("v").split(".")
    return ["v" + ".".join(parts[:i]) for i in range(len(parts), 0, -1)]

print(semver_aliases("v2.0.0"))

release_200 = b"backend release 2.0.0"
for tag in semver_aliases("v2.0.0"):
    push(f"backend:{tag}", release_200)
```

Slicing `parts[:i]` for `i = 3, 2, 1` produces the ladder. Note what pushing v2.0.0 did **not** touch: `v1` still points at the 1.x line — major-version tags only move within their own major, which is the entire promise of semver tagging.
</details>

## 8. ⚠️ Common pitfalls

Four classics from the two chapters — each one a production incident wearing a trench coat:

| Pitfall | Symptom | Fix |
|---|---|---|
| **Deploying mutable `:latest`** | two servers pull an hour apart and run *different code*; or "it redeployed but nothing changed" because a stale `:latest` was cached | pin digests or SHA/semver tags for anything that must be reproducible; keep `:latest` as a convenience pointer only — and keep SHA tags around for rollback |
| **Echoing secrets in `run:` steps** | a token or SSH key lands in the build logs; GitHub masks known secret *values*, but not transformed (e.g. base64'd) or hardcoded ones | never `echo` a secret, not even "for debugging"; pass secrets as action inputs / env; rotate any key that ever touched a log |
| **Green locally, red in CI** | "works on my machine" — a package you installed months ago was never added to `requirements.txt`, and the fresh runner is a *blank machine* | treat the runner's amnesia as a feature: pin every dependency; when CI is red and your laptop is green, believe CI |
| **Never caching** | every run spends minutes re-downloading identical wheels; feedback slows down and people stop waiting for CI | `cache: pip` on `setup-python`, or `actions/cache` keyed on `hashFiles('requirements.txt')` — you built the key logic in §6 |

## 🧪 Practice exercises

Everything below builds on objects already in memory: `wf`, `DEMO_WF`, `REPO`, `run_workflow`, `execution_waves`, `REGISTRY`, `push`…

### Exercise 1 — ⭐ Pin-spotting

Best practice from the chapter: **pin action versions** — never float on someone else's `@main`. Write `action_pins(workflow)` that returns a dict mapping each action to its pinned version for every `uses:` step, and run it on the real workflow `wf`. You should find **6 distinct actions**. Which one is *not* maintained by GitHub or Docker (and therefore deserves the most scrutiny before you trust it with secrets)?

In [21]:
# Your code here  👇

<details>
<summary>💡 <b>Solution</b></summary>

```python
def action_pins(workflow: dict) -> dict:
    pins = {}
    for job in workflow["jobs"].values():
        for step in job.get("steps", []):
            if "uses" in step:
                action, _, version = step["uses"].partition("@")
                pins[action] = version
    return pins

for action, version in action_pins(wf).items():
    print(f"   {action:<32} @{version}")
```

Six actions: `actions/checkout@v4`, `actions/setup-python@v5`, `docker/setup-buildx-action@v3`, `docker/login-action@v3`, `docker/build-push-action@v6`, and `appleboy/ssh-action@v1`. The last one is a *community* action — it receives your server's SSH key as an input, so pinning it (ideally to a full commit SHA) matters most: a hijacked update to that action could exfiltrate the key.
</details>

### Exercise 2 — ⭐⭐ Grow the factory: a parallel lint job

Add a `lint` job to a **copy** of `DEMO_WF` (use `copy.deepcopy` — don't mutate the original): one step running `python -m compileall -q .` (a cheap "does it even parse?" check). Make `build` need **both** `test` and `lint`. Predict the waves first, then confirm with `execution_waves` and run the whole thing with `run_workflow`. Did total pipeline time grow by the lint job's duration — and *would* it, on GitHub?

In [22]:
# Your code here  👇

<details>
<summary>💡 <b>Solution</b></summary>

```python
import copy

wf_lint = copy.deepcopy(DEMO_WF)
wf_lint["jobs"]["lint"] = {
    "steps": [{"name": "Compile check", "run": "python -m compileall -q ."}],
}
wf_lint["jobs"]["build"]["needs"] = ["test", "lint"]

lint_needs = {j: as_list(s.get("needs")) for j, s in wf_lint["jobs"].items()}
print("waves:", execution_waves(lint_needs))

table, st = run_workflow(wf_lint, REPO)
table
```

Waves become `[['lint', 'test'], ['build'], ['deploy']]` — lint joins wave 1 with **no** `needs:` edge, so on GitHub it would run *in parallel* with `test` on its own fresh runner and add ~zero wall time. Our toy runner executes waves sequentially (one laptop, one worker), so locally you do pay for it — that gap is exactly what Stretch exercise A closes.
</details>

### Exercise 3 — ⭐⭐ Cache fallbacks with `restore-keys`

Real `actions/cache` takes an optional `restore-keys:` list — **key prefixes** tried in order when the exact key misses. A prefix match is a *partial hit*: you restore an older cache (most wheels already there) and `pip install` only tops it up. Implement:

```python
lookup(cache, key, restore_keys) -> tuple[str, str | None]
```

returning `("exact", key)`, `("partial", <newest key matching a prefix>)`, or `("miss", None)`. Treat "newest" as "inserted latest" (dicts preserve insertion order). Test all three outcomes.

In [23]:
# Your code here  👇

<details>
<summary>💡 <b>Solution</b></summary>

```python
def lookup(cache: dict, key: str, restore_keys: tuple = ()) -> tuple:
    if key in cache:
        return "exact", key
    for prefix in restore_keys:
        matches = [k for k in cache if k.startswith(prefix)]
        if matches:
            return "partial", matches[-1]     # insertion order → last = newest
    return "miss", None

demo_cache = {"pip-aaa111": "…", "pip-bbb222": "…", "npm-ccc333": "…"}
print(lookup(demo_cache, "pip-bbb222", ("pip-",)))   # ('exact', 'pip-bbb222')
print(lookup(demo_cache, "pip-zzz999", ("pip-",)))   # ('partial', 'pip-bbb222')
print(lookup(demo_cache, "pip-zzz999"))              # ('miss', None)
```

This is why changing one line of `requirements.txt` doesn't cost you the whole cache in a well-configured workflow: the exact key misses, but `restore-keys: pip-` rescues the previous run's packages, and only the delta is downloaded.
</details>

### Exercise 4 — ⭐⭐ Debug me 🐞

A teammate "simplified" `run_step` — and now the pipeline **crashes**, and (worse) it **declared a broken step green** just before crashing. Run the cell, read the traceback, and find **both** bugs. The chapter's exit-code rule is your compass.

In [24]:
# ⚠️ THIS CELL INTENTIONALLY ERRORS — find the TWO bugs before opening the solution.
def run_step_buggy(cmd: str, cwd) -> str:
    proc = subprocess.run(re.sub(r"^python\b", PY, cmd), shell=True, cwd=cwd)
    print("step finished ✅ (…did it, though?)")
    return proc.stdout.strip()

run_step_buggy('python -c "import nonexistent_module"', REPO)

step finished ✅ (…did it, though?)


Traceback (most recent call last):
  File "<string>", line 1, in <module>
ModuleNotFoundError: No module named 'nonexistent_module'


AttributeError: 'NoneType' object has no attribute 'strip'

<details>
<summary>💡 <b>Solution</b></summary>

**Bug 1 — the success message lies.** The child process exited with code `1` (`ModuleNotFoundError` — you can see its traceback in the output above), but nothing ever checks `proc.returncode`, so the step is announced green. In CI **the exit code *is* the contract**; a runner that ignores it ships broken code with a smile.

**Bug 2 — `proc.stdout` is `None`.** Without `capture_output=True, text=True`, `subprocess.run` doesn't collect the output at all (it flows straight to the terminal — that's why the child's traceback appeared above). So `proc.stdout` is `None`, and `.strip()` explodes with `AttributeError`.

```python
def run_step_fixed(cmd: str, cwd) -> dict:
    proc = subprocess.run(re.sub(r"^python\b", PY, cmd), shell=True, cwd=cwd,
                          capture_output=True, text=True)
    ok = proc.returncode == 0
    print("✅ step succeeded" if ok else f"❌ step failed (exit {proc.returncode})")
    return {"exit": proc.returncode, "log": (proc.stdout + proc.stderr).strip()}

print(run_step_fixed('python -c "import nonexistent_module"', REPO)["exit"])   # → 1, honestly
```

Same lesson as the big runner in §4: *capture the log, check the code, never infer success from the absence of an exception.*
</details>

## 🧠 Stretch exercises

### Stretch exercise A — ⭐⭐⭐ Run the waves for real (in parallel)

Our runner executes each wave's jobs one after another; GitHub runs them **concurrently on separate machines**. Close the gap: write `run_workflow_parallel(workflow, repo_dir)` that uses `concurrent.futures.ThreadPoolExecutor` to run all jobs of a wave at the same time (threads are fine — the work happens in subprocesses, which release the GIL). Prove the speed-up with a workflow of two 1-second `sleep` jobs feeding one final job: ~1 s per wave instead of ~2 s.

In [25]:
# Your code here  👇

<details>
<summary>💡 <b>Solution</b></summary>

```python
from concurrent.futures import ThreadPoolExecutor

def run_job(jobs: dict, job_id: str, repo_dir: Path):
    for step in jobs[job_id]["steps"]:
        res = run_step(step["run"], repo_dir)
        if res["exit"] != 0:
            return "failure"
    return "success"

def run_workflow_parallel(workflow: dict, repo_dir: Path) -> dict:
    jobs = workflow["jobs"]
    needs = {j: as_list(s.get("needs")) for j, s in jobs.items()}
    status = {}
    t0 = time.perf_counter()
    for wave in execution_waves(needs):
        runnable = [j for j in wave if all(status.get(d) == "success" for d in needs[j])]
        for j in wave:
            if j not in runnable:
                status[j] = "skipped"
        with ThreadPoolExecutor(max_workers=len(runnable) or 1) as pool:
            for job_id, st in zip(runnable, pool.map(lambda j: run_job(jobs, j, repo_dir), runnable)):
                status[job_id] = st
    print(f"wall time: {time.perf_counter() - t0:.2f}s   {status}")
    return status

SLOW = {"jobs": {
    "a":    {"steps": [{"name": "sleep", "run": "python -c \"import time; time.sleep(1)\""}]},
    "b":    {"steps": [{"name": "sleep", "run": "python -c \"import time; time.sleep(1)\""}]},
    "ship": {"needs": ["a", "b"], "steps": [{"name": "done", "run": "python -c \"print('shipped')\""}]},
}}
run_workflow_parallel(SLOW, REPO)     # wave 1 ≈ 1s (a ∥ b), not ≈ 2s
```

The wave structure did all the hard thinking already — parallelism is just "map over the wave". This is also why GitHub can charge per job-minute and still feel fast: the DAG *is* the scheduler.
</details>

### Stretch exercise B — ⭐⭐⭐ A workflow linter

Turn §8 into software. Write `lint_workflow(workflow)` that returns a list of findings, checking at least: **(1)** any deploy-ish job (its id contains `deploy`, `push`, `publish` or `release`) without an `if:` guard mentioning `refs/heads/main` — it would run on PRs; **(2)** any deploy-ish job with no `needs:` — it may ship untested code; **(3)** any `uses:` without an `@` version pin; **(4)** any `run:` step that `echo`es something from `secrets.`. The real workflow `wf` should come back clean; craft a small bad workflow that trips all four rules.

In [26]:
# Your code here  👇

<details>
<summary>💡 <b>Solution</b></summary>

```python
def lint_workflow(workflow: dict) -> list[str]:
    findings = []
    for job_id, spec in workflow["jobs"].items():
        deployish = any(w in job_id for w in ("deploy", "push", "publish", "release"))
        if deployish and "refs/heads/main" not in spec.get("if", ""):
            findings.append(f"{job_id}: deploy-ish job without a main-branch `if:` guard — it will run on PRs")
        if deployish and not as_list(spec.get("needs")):
            findings.append(f"{job_id}: no `needs:` — may ship code the tests never validated")
        for step in spec.get("steps", []):
            uses = step.get("uses", "")
            if uses and "@" not in uses:
                findings.append(f"{job_id}: unpinned action {uses!r} — pin @vN or a commit SHA")
            run_cmd = step.get("run", "")
            if "echo" in run_cmd and "secrets." in run_cmd:
                findings.append(f"{job_id}: a `run:` step echoes a secret — logs are forever")
    return findings

print(lint_workflow(wf) or "✅ real workflow: no findings")

bad = {"jobs": {"deploy": {"steps": [
    {"uses": "someone/action"},
    {"run": "echo ${{ secrets.SERVER_SSH_KEY }}"},
]}}}
print(*lint_workflow(bad), sep="\n")
```

Congratulations — you've re-invented the core of tools like `actionlint` and `zizmor`. Every rule here is a real incident class: accidental PR deploys, untested ships, supply-chain action hijacks, and leaked keys.
</details>

### Stretch exercise C — ⭐⭐⭐ Registry retention policy

`registries.md`'s best practice: pushing a SHA tag on every commit means **images pile up** — set retention rules. Write `prune(registry, blobs, keep_sha=2)` that: deletes all but the newest `keep_sha` *SHA-style* tags (tag part is 7–40 hex chars), **never** touches `:latest` or semver tags, and then garbage-collects blobs no tag references any more. Run it on the toy `REGISTRY`/`BLOBS` and report what survived.

In [27]:
# Your code here  👇

<details>
<summary>💡 <b>Solution</b></summary>

```python
def prune(registry: dict, blobs: dict, keep_sha: int = 2) -> None:
    sha_tags = [t for t in registry if re.fullmatch(r".+:[0-9a-f]{7,40}", t)]
    doomed = sha_tags[:-keep_sha] if keep_sha else sha_tags
    for tag in doomed:
        print(f"   deleting tag  {tag}")
        del registry[tag]
    referenced = set(registry.values())
    for digest in [d for d in blobs if d not in referenced]:
        print(f"   gc blob       {digest[:22]}…")
        del blobs[digest]

prune(REGISTRY, BLOBS, keep_sha=1)
print(f"\n{len(REGISTRY)} tags and {len(BLOBS)} blobs survive")
```

The regex keeps `backend:latest` and `backend:v1.2.3` safe (`latest` and `v1.2.3` aren't pure hex) while catching `backend:e83c5f2`. The two-phase shape — drop tags, then collect unreferenced blobs — is exactly how real registries reclaim space (GHCR retention rules, `docker system prune`, ACR purge tasks), and it's why our deploy job runs `docker image prune -f` on the server.
</details>

## 🎁 Bonus mini-project — `act` at home: one-command CI

The real tool [`act`](https://github.com/nektos/act) replays GitHub Actions workflows on your laptop in Docker. Build the paper-airplane version from parts you already own. One function:

```python
ci(repo_dir, workflow, ref="refs/heads/main") -> str
```

that replays the entire lab end to end:

1. **Event** — compute a fake commit SHA (hash the source files) and announce the push: the loading-dock bell.
2. **Cache** — compute the pip cache key with `hash_files` and report hit/miss against `CACHE`.
3. **CI** — run the workflow with `run_workflow` and print the results table.
4. **CD** — only if *every* job succeeded: `push` the built `dist/app.bundle` to the toy registry as `backend:latest` **and** `backend:<sha>` (the module's dual-tag scheme).
5. Return `"success"` or `"failure"`.

Then call it twice — once on `main` (full CI→CD) and once with a PR ref (tested, but nothing ships) — and check `REGISTRY` afterwards to confirm the PR run pushed nothing.

In [28]:
# Your code here  👇

<details>
<summary>💡 <b>Solution sketch</b></summary>

```python
def ci(repo_dir: Path, workflow: dict, ref: str = "refs/heads/main") -> str:
    print("=" * 62)
    sha = hashlib.sha256((repo_dir / "calculator.py").read_bytes()).hexdigest()[:7]
    print(f"🔔 push {sha} on {ref} — matches on: {list(workflow['on'])}")

    restore_or_save(CACHE, "pip-" + hash_files(repo_dir / "requirements.txt"))

    table, status = run_workflow(workflow, repo_dir, ref=ref, verbose=False)
    print(table.to_string(index=False))

    if all(s == "success" for s in status.values()):
        image = (repo_dir / "dist" / "app.bundle").read_bytes()
        push("backend:latest", image)
        push(f"backend:{sha}", image)
        print(f"🚚 delivered: backend:latest + backend:{sha}")
        return "success"
    print("🛑 pipeline stopped — nothing was pushed")
    return "failure"

ci(REPO, DEMO_WF)                             # main: tested → built → pushed
ci(REPO, DEMO_WF, ref="refs/pull/7/merge")    # PR:   tested → guards stop the truck
```

That's the whole chapter in one function call: the event rings the bell, the cache saves minutes, the DAG gates the stages, the `if:` guards protect production, and the registry ledger records exactly what shipped. When you later watch a real run in the Actions tab, you'll know precisely what every line of that UI is doing — because you've written it.
</details>

## 🧠 Key takeaways

> 🧭 **The story in one line.** We demystified the automated factory: a workflow is a YAML-shaped program (events ring the bell, jobs form a DAG, steps are shell commands), and we rebuilt the machine that runs it in ~80 lines — topological waves, exit codes, skip propagation, matrix products, hash-keyed caches, and a warehouse ledger where `:latest` moves but digests never do.

1. **A workflow is a YAML-shaped program**: `on:` events → `jobs:` (a DAG) → `steps:` (shell commands). The runner is a loop — you wrote it.
2. **Two kinds of step**: `uses:` plugs in a pre-built power tool (pin its version!); `run:` is your own two hands.
3. **`needs:` edges make the DAG.** Jobs are parallel by default; waves fall out of a topological sort; wall time = slowest job per wave.
4. **The exit code is the whole contract.** First non-zero step fails the job; dependents are skipped; broken code is never built, let alone deployed.
5. **`if: github.ref == 'refs/heads/main'` guards the risky jobs** — PRs get tested, only `main` builds and deploys.
6. **A matrix is a cartesian product** of configurations; `fail-fast` cancels the siblings once one universe turns red.
7. **The runner is a blank, amnesiac machine** — pin every dependency, and cache with a key like `pip-${{ hashFiles('requirements.txt') }}` that invalidates itself when (and only when) the deps change.
8. **Tags move; digests don't.** Push `:latest` for convenience *plus* `:<commit-sha>` for rollback and audit; semver ladders (`v1.2.3` / `v1.2` / `v1`) let consumers choose their risk.

## ✅ Self-assessment

- [ ] Name the six nouns of GitHub Actions (workflow, event, job, step, action, runner) and give the factory analogy for each
- [ ] Point at any step in `ci-cd.yml` and say whether it's `uses:` or `run:` — and why the `@v4` pin matters
- [ ] Extract the `needs:` edges from a workflow and hand-compute the execution waves
- [ ] Explain why a failing step stops its job, and which jobs get skipped afterwards
- [ ] Expand a `strategy.matrix` by hand and say how many jobs it creates — and what `fail-fast` changes
- [ ] Compute a cache key with `hashFiles`-style hashing and predict hit vs miss after editing `requirements.txt`
- [ ] Explain to a teammate why deploying `:latest` is risky, and what the `:latest` + `:<sha>` dual-tag scheme buys

## 🚀 Next step

Your pipeline "pushed" images to a toy warehouse — the next lab picks them up on the other side. Continue with **`lab03_deploy_dns_https_monitoring.ipynb`**, the hands-on companion to the `deployment.md`, `dns.md`, `https.md` and `monitoring.md` chapters: a server pulls the images, a domain name finds the server, TLS locks the connection, and monitoring tells you it's still alive at 3 a.m.

And when you want the real thing: fork the course repo, copy `example-app/.github/workflows/ci-cd.yml` to the **repo-root** `.github/workflows/` (the §2 path caveat), push — and watch the factory you now fully understand run itself.

> 🚀 You didn't just read the assembly line's manual — you built the conveyor belt. Every green check mark you see from now on is a loop you could have written.